# song2nam: 음원 → .nam

곡 음원 하나와 세션 정보(기타·앰프·캐비닛·부스트)만으로 NAM에서 바로 쓸 수 있는 `.nam` 캡처를 만듭니다.

**실행 전에:** [런타임 → 런타임 유형 변경]에서 **T4 GPU**를 고르세요.

순서대로 각 셀의 ▶를 누르면 됩니다. 코드는 숨겨져 있고, 오른쪽 폼에서 값만 고르면 됩니다.

1. 준비 (처음 실행 시 데이터 내려받기 포함, 몇 분)
2. 음원 올리기
3. 세션 정보 고르기 (드롭다운): 각 항목의 뜻은 [세션 정보 입력 안내](https://github.com/gmeag01/song2nam/blob/main/docs/session-guide.md)를 보세요
4. `.nam` 만들기 (T4 기준 약 25분)
5. 결과 받기

In [ ]:
#@title 1. 준비
#@markdown Drive를 쓰면 내려받은 데이터와 결과가 `MyDrive/song2nam/`에 남아, 다음 실행부터 내려받기를 건너뜁니다.
use_drive = True #@param {type:"boolean"}

import os, subprocess
REPO = '/content/song2nam'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
os.chdir(REPO)

if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/song2nam'
    for d in ['data', 'runs']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
        if not os.path.islink(d):
            subprocess.run(['rm', '-rf', d])
            os.symlink(f'{DRIVE}/{d}', d)
os.makedirs('data/songs', exist_ok=True)

# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
subprocess.run("grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt",
               shell=True, check=True)

from ampcap import get_data
get_data.main(['--data', 'data'])

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (런타임 유형을 T4 GPU로 바꾸세요)')

In [ ]:
#@title 2. 음원 올리기
#@markdown 비워 두고 실행하면 파일 선택 창이 뜹니다. 이미 올린 파일을 쓰려면 경로를 적으세요 (예: `data/songs/my_song.mp3`).
song_path = "" #@param {type:"string"}

import os, shutil
if not song_path:
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit('파일을 고르지 않았습니다. 다시 실행하세요.')
    name = next(iter(up))
    song_path = f'data/songs/{name}'
    shutil.move(name, song_path)
if not os.path.exists(song_path):
    raise FileNotFoundError(song_path)
SONG = song_path
print('음원:', SONG, f'{os.path.getsize(SONG) / 1e6:.1f} MB')

In [ ]:
#@title 3. 세션 정보
#@markdown 모르는 항목은 **모름**으로 두세요. 틀린 값보다 낫습니다. 각 항목의 뜻은 [세션 정보 입력 안내](https://github.com/gmeag01/song2nam/blob/main/docs/session-guide.md)에 있습니다.
#@markdown
#@markdown **출처**는 그 정보를 어디서 알았는지입니다. 짐작일수록 음원 분석에 더 맡깁니다.
#@markdown ### 곡
artist = "" #@param {type:"string"}
song = "" #@param {type:"string"}
#@markdown 곡의 기타를 모두 같은 장비로 녹음했으면 체크 (기타리스트가 둘 이상이면 해제)
same_rig = False #@param {type:"boolean"}
#@markdown ### 기타 픽업
pickup = "모름" #@param ["모름", "싱글 (Strat, Tele)", "P90 (Les Paul Junior, SG Special)", "험버커 (Les Paul, SG, 메탈 기타)"]
pickup_source = "확실 (그 곡 인터뷰·크레딧)" #@param ["확실 (그 곡 인터뷰·크레딧)", "아마 (평소 장비, 기어 사이트)", "짐작"]
#@markdown ### 앰프
#@markdown 계열을 **모름**으로 두고 앰프 이름을 적으면 이름에서 계열을 찾습니다 (예: JCM800, AC30, Rectifier).
amp_family = "모름" #@param ["모름", "Fender 계열 (Twin, Deluxe, Bassman)", "Vox 계열 (AC30, AC15)", "Marshall·Orange 계열 (JCM800, Plexi)", "미국 하이게인 (Mesa, 5150, Soldano)", "모던 하이게인 (Diezel, ENGL, Friedman)"]
amp_name = "" #@param {type:"string"}
#@markdown 게인은 장비가 아니라 **귀로** 고르세요.
amp_gain = "모름" #@param ["모름", "클린 (세게 칠 때만 살짝 깨짐)", "크런치 (코드 음이 하나하나 들림)", "하이게인 (서스테인 길고 메탈 톤)"]
amp_source = "확실 (그 곡 인터뷰·크레딧)" #@param ["확실 (그 곡 인터뷰·크레딧)", "아마 (평소 장비, 기어 사이트)", "짐작"]
#@markdown ### 캐비닛
cab_size = "모름" #@param ["모름", "1x12 (작은 콤보)", "2x12 (큰 콤보)", "4x12 (스택)"]
cab_source = "짐작" #@param ["확실 (그 곡 인터뷰·크레딧)", "아마 (평소 장비, 기어 사이트)", "짐작"]
#@markdown ### 부스트 (앰프 앞 오버드라이브 페달)
boost = "모름" #@param ["모름", "없음", "오버드라이브 페달 (Tube Screamer 계열)"]
boost_source = "짐작" #@param ["확실 (그 곡 인터뷰·크레딧)", "아마 (평소 장비, 기어 사이트)", "짐작"]
#@markdown ### 구간
#@markdown 기타 한 대만 나오는 구간을 `분:초-분:초`로, 여러 개면 쉼표로 (예: `0:20-0:45, 2:10-2:22`). 비우면 자동 추천 구간을 씁니다.
segments = "" #@param {type:"string"}

# 화면 글자 → 세션 값 (위 #@param 목록과 같은 순서)
PICKUP = dict(zip(["모름", "싱글 (Strat, Tele)", "P90 (Les Paul Junior, SG Special)", "험버커 (Les Paul, SG, 메탈 기타)"],
                  ["unknown", "single", "p90", "humbucker"]))
FAMILY = dict(zip(["모름", "Fender 계열 (Twin, Deluxe, Bassman)", "Vox 계열 (AC30, AC15)", "Marshall·Orange 계열 (JCM800, Plexi)",
                   "미국 하이게인 (Mesa, 5150, Soldano)", "모던 하이게인 (Diezel, ENGL, Friedman)"],
                  ["unknown", "fender", "vox", "british", "american_hg", "modern_hg"]))
GAIN = dict(zip(["모름", "클린 (세게 칠 때만 살짝 깨짐)", "크런치 (코드 음이 하나하나 들림)", "하이게인 (서스테인 길고 메탈 톤)"],
                ["unknown", "low", "mid", "high"]))
CAB = dict(zip(["모름", "1x12 (작은 콤보)", "2x12 (큰 콤보)", "4x12 (스택)"], ["unknown", "1x12", "2x12", "4x12"]))
BOOST = dict(zip(["모름", "없음", "오버드라이브 페달 (Tube Screamer 계열)"], ["unknown", "none", "tube_screamer"]))
SOURCE = dict(zip(["확실 (그 곡 인터뷰·크레딧)", "아마 (평소 장비, 기어 사이트)", "짐작"], [1.0, 0.5, 0.3]))

import json, os
import torch
from ampcap import session as S
from ampcap.graybox.amp import denormalize
session = {
    "artist": artist, "song": song, "same_rig": same_rig,
    "guitar": {"pickup": PICKUP[pickup]},
    "amp": {"family": FAMILY[amp_family], "model": amp_name, "gain": GAIN[amp_gain]},
    "cab": {"size": CAB[cab_size]},
    "boost": BOOST[boost],
    "confidence": {"guitar": SOURCE[pickup_source], "amp": SOURCE[amp_source],
                   "cab": SOURCE[cab_source], "boost": SOURCE[boost_source]},
}
checked = S.validate(session)  # 잘못된 값이면 여기서 멈춤
SESSION = os.path.splitext(SONG)[0] + '.session.json'
with open(SESSION, 'w', encoding='utf-8') as f:
    json.dump(session, f, ensure_ascii=False, indent=2)
def to_sec(t):  # "2:10" → 130.0, "1:05.5" → 65.5, "45" → 45.0
    m, _, sec = t.strip().rpartition(':')
    return int(m or 0) * 60 + float(sec)
spans = []
for part in filter(str.strip, segments.split(',')):
    try:
        a, b = (to_sec(t) for t in part.split('-'))
    except ValueError:
        raise ValueError(f'구간 {part.strip()!r}을 읽지 못했습니다. 분:초-분:초 형식으로 적으세요 (예: 0:20-0:45).')
    if b <= a:
        raise ValueError(f'구간 {part.strip()!r}: 끝이 시작보다 앞입니다.')
    spans.append(f'{a:g}:{b:g}')
SEGMENTS = ','.join(spans) or None  # pipeline --segments 형식 (초)

# 고른 내용 요약
fam = checked['amp']['family']
by_name = S.model_to_family(amp_name)
if FAMILY[amp_family] == 'unknown' and fam != 'unknown':
    print(f'앰프 이름 {amp_name!r}에서 계열을 찾았습니다.')
elif by_name and FAMILY[amp_family] != 'unknown' and by_name != fam:
    print(f'⚠ 앰프 이름 {amp_name!r}은 보통 다른 계열입니다. 앰프 계열 선택을 확인하세요.')
label = {v: k.split(' (')[0] for d in (PICKUP, FAMILY, GAIN, CAB, BOOST) for k, v in d.items() if v != 'unknown'}
label['tube_screamer'] = '오버드라이브 페달'
label['none'] = '부스트 없음'
parts = [label[v] for v in (fam, checked['amp']['gain'], checked['guitar']['pickup'],
                            checked['cab']['size'], checked['boost']) if v != 'unknown']
prior = S.compute_prior(checked)
lo, hi = denormalize(torch.tensor(prior.lo)), denormalize(torch.tensor(prior.hi))
print('톤 찾기 조건:', ' · '.join(parts) if parts else '정보 없음 (음원만 보고 찾습니다)')
print(f"프리앰프 게인 범위: 1단 {lo['pre1_gain_db']:.0f}~{hi['pre1_gain_db']:.0f} dB, "
      f"2단 {lo['pre2_gain_db']:.0f}~{hi['pre2_gain_db']:.0f} dB")
for w in prior.warnings:
    print('참고:', w)
print('저장:', SESSION, '| 구간:', SEGMENTS or '자동 추천')

In [ ]:
#@title 4. .nam 만들기
#@markdown 톤 찾기(⑥) 스텝 수와 NAM 학습 epoch 수. 기본값은 T4에서 약 25분입니다.
steps = 800 #@param {type:"integer"}
epochs = 100 #@param {type:"integer"}

import os, re
from ampcap import pipeline
label = '_'.join(x for x in [artist, song] if x) or os.path.splitext(os.path.basename(SONG))[0]
RUN_DIR = str(pipeline.make_run_dir(label))
args = [SONG, '--session', SESSION, '--egdb', 'data/EGDB/audio_DI', '--nam-input', 'data/nam/v3_0_0.wav',
        '--steps', str(steps), '--epochs', str(epochs), '--run-dir', RUN_DIR]
if SEGMENTS:
    args += ['--segments', SEGMENTS]
pipeline.main(args)

In [ ]:
#@title 5. 결과 받기
#@markdown `model.nam`을 내려받습니다. NAM 플러그인에서 쓸 때는 **IR 로더를 끄세요** (캐비닛이 이미 들어 있습니다).
download = True #@param {type:"boolean"}

import json, os
from IPython.display import Image, Audio, display
d = json.load(open(f'{RUN_DIR}/distill.json'))
lvl, ver = d['input_level'], d['verify']
print(f"G와 .nam 차이 (ESR): {ver['esr_mean']:.4f}  ({'좋음' if ver['ok'] else '높음: epochs를 늘려 다시 실행해 보세요'})")
print(f"DI 트랙 연주 RMS를 {lvl['di_ref_rms_dbfs']:.1f} dBFS 근처에 맞추면 톤 찾기 때와 같은 드라이브가 걸립니다.")
seg = json.load(open(f'{RUN_DIR}/segments.json'))
if seg.get('warnings'):
    print('구간 경고:', seg['warnings'])
display(Image(f'{RUN_DIR}/loss_curve.png'))
print('원곡 기타 스템 / 후보 0 (θ*, EGDB DI를 통과시킨 소리)')
display(Audio(f'{RUN_DIR}/stem.wav'))
display(Audio(f'{RUN_DIR}/candidate_0.wav'))
print('결과 폴더:', RUN_DIR)
if download:
    from google.colab import files
    files.download(f'{RUN_DIR}/model.nam')